# Gradio Demo: So sánh Stage 3 PPO với Stage 2 SFT

Notebook này dựng giao diện Gradio để nhập một bài báo tiếng Việt và sinh tóm tắt song song bằng 2 model:

- **Stage 2 SFT checkpoint**: `NLP_Final/Output stage 2 vit5 checkpoint/best_model`
- **Stage 3 PPO**: `NLP_Final/ppo_run/runs/ppo_run/best_model` hoặc các biến thể path tương đương

Notebook có cơ chế tự dò path khi chạy trên Colab sau khi mount Drive, và cũng chạy được local nếu workspace có `best_model/` và `ppo_run-*/ppo_run/best_model/`.

## 1. Cài package nếu cần

Nếu runtime mới chưa có thư viện, đặt `INSTALL_DEPS = True`, chạy cell này, rồi restart runtime trước khi chạy tiếp.

In [ ]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    import sys, subprocess
    packages = [
        "gradio==4.44.1",
        "transformers==4.46.3",
        "sentencepiece==0.2.0",
        "safetensors==0.4.5",
        "accelerate==1.1.1",
        "pandas",
        "torch",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])


## 2. Mount Drive và dò đường dẫn model

Nếu notebook không tự tìm đúng folder, sửa trực tiếp `STAGE2_MODEL_DIR` và `STAGE3_MODEL_DIR` ở cuối cell.

In [ ]:
from pathlib import Path
import os

MOUNT_DRIVE = True

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "best_model").exists() and (PROJECT_ROOT.parent / "best_model").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DRIVE_ROOT = Path("/content/drive/MyDrive/NLP_Final")

def first_existing(candidates):
    for path in candidates:
        path = Path(path)
        if (path / "config.json").exists() and (path / "model.safetensors").exists():
            return path
    return Path(candidates[0])

local_stage3 = sorted(PROJECT_ROOT.glob("ppo_run*/ppo_run/best_model"))

stage2_candidates = [
    DRIVE_ROOT / "Output stage 2 vit5 checkpoint" / "best_model",
    DRIVE_ROOT / "Output stage 2 vit5 checkpoint" / "model_for_rl",
    PROJECT_ROOT / "best_model",
]

stage3_candidates = [
    DRIVE_ROOT / "ppo_run" / "runs" / "ppo_run" / "best_model",
    DRIVE_ROOT / "ppo_run" / "ppo_run" / "best_model",
    DRIVE_ROOT / "ppo_run" / "best_model",
    DRIVE_ROOT / "ViT5_stage_3" / "runs" / "ppo_run" / "best_model",
    *(local_stage3 if local_stage3 else []),
]

STAGE2_MODEL_DIR = first_existing(stage2_candidates)
STAGE3_MODEL_DIR = first_existing(stage3_candidates)

data_candidates = [
    Path("/content/drive/MyDrive/data/test/dataset.jsonl"),
    Path("/content/drive/MyDrive/NLP_Final/data/test/dataset.jsonl"),
    PROJECT_ROOT / "data" / "test" / "dataset.jsonl",
    PROJECT_ROOT / "data" / "val" / "dataset.jsonl",
]
DATA_PATH = next((p for p in data_candidates if p.exists()), data_candidates[0])

print("Stage 2 SFT:", STAGE2_MODEL_DIR)
print("Stage 3 PPO:", STAGE3_MODEL_DIR)
print("Data path  :", DATA_PATH if DATA_PATH.exists() else "khong tim thay, van co the dan text thu cong")

missing = []
for label, model_dir in [("Stage 2", STAGE2_MODEL_DIR), ("Stage 3", STAGE3_MODEL_DIR)]:
    if not (model_dir / "config.json").exists() or not (model_dir / "model.safetensors").exists():
        missing.append(f"{label}: {model_dir}")

if missing:
    raise FileNotFoundError("Khong tim thay model hop le:\n" + "\n".join(missing))


## 3. Imports, metric helpers và dữ liệu mẫu

In [ ]:
import gc
import json
import random
import re
import time
from functools import lru_cache

import gradio as gr
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MAX_INPUT_LEN = 1024
MAX_TARGET_LEN = 384
NO_REPEAT_NGRAM = 3
TARGET_RATIO = 0.225
NUM_RE = re.compile(r"\d+(?:[.,]\d+)*")

def normalize_text(text):
    return " ".join((text or "").split())

def extract_numbers(text):
    return set(NUM_RE.findall(text or ""))

def length_ratio(source, pred):
    return len(normalize_text(pred).split()) / max(len(normalize_text(source).split()), 1)

def length_score(source, pred, target=TARGET_RATIO):
    ratio = length_ratio(source, pred)
    if ratio <= target:
        return max(0.0, 1.0 - (target - ratio) / target)
    return max(0.0, 1.0 - 1.5 * (ratio - target) / target)

def number_recall(source, pred):
    src = extract_numbers(source)
    if not src:
        return 1.0
    return len(src & extract_numbers(pred)) / len(src)

def number_hallucination(source, pred):
    src = extract_numbers(source)
    out = extract_numbers(pred)
    if not out:
        return 0.0
    return len(out - src) / len(out)

def quality_score(source, pred):
    return 0.55 * number_recall(source, pred) + 0.35 * length_score(source, pred) - 0.55 * number_hallucination(source, pred)

def metric_row(name, source, summary, elapsed):
    src_nums = extract_numbers(source)
    out_nums = extract_numbers(summary)
    return {
        "model": name,
        "words": len(normalize_text(summary).split()),
        "length_ratio": round(length_ratio(source, summary), 4),
        "number_recall": round(number_recall(source, summary), 4),
        "number_hallucination": round(number_hallucination(source, summary), 4),
        "kept_numbers": f"{len(src_nums & out_nums)}/{len(src_nums)}",
        "new_numbers": len(out_nums - src_nums),
        "score": round(quality_score(source, summary), 4),
        "time_sec": round(elapsed, 2),
    }

def numbers_comparison(source, sft_summary, ppo_summary):
    src = extract_numbers(source)
    sft = extract_numbers(sft_summary)
    ppo = extract_numbers(ppo_summary)
    universe = sorted(src | sft | ppo)
    rows = []
    for num in universe[:120]:
        rows.append([
            num,
            "yes" if num in src else "no",
            "yes" if num in sft else "no",
            "yes" if num in ppo else "no",
        ])
    return rows if rows else [["-", "no numbers", "-", "-"]]

def pick_field(row, names):
    for name in names:
        value = row.get(name)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def load_examples(path=DATA_PATH, limit=12):
    if not Path(path).exists():
        return [], {}
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    rng = random.Random(42)
    rng.shuffle(rows)
    choices, mapping = [], {}
    for i, row in enumerate(rows[:limit], 1):
        source = pick_field(row, ["source", "text", "article", "document", "content", "src", "input"])
        if not source:
            continue
        title = pick_field(row, ["title", "id"]) or f"Mau {i}"
        reference = pick_field(row, ["summary", "target", "abstract", "tgt", "label"])
        label = f"{i}. {title[:90]}"
        choices.append(label)
        mapping[label] = {"source": source, "reference": reference}
    return choices, mapping

EXAMPLE_CHOICES, EXAMPLE_MAP = load_examples()
EXAMPLE_CHOICES[:3]


## 4. Load model và hàm sinh tóm tắt

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE, "| dtype:", DTYPE)

def load_tokenizer(model_dir):
    # use_fast=True avoids KeyError: 0 bug in transformers 4.46.x convert_to_native_format
    # which tries vocab[0] (int key) but ViT5/T5 vocab has string keys
    try:
        return AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)
    except Exception:
        try:
            return AutoTokenizer.from_pretrained(str(model_dir), use_fast=False, legacy=False)
        except TypeError:
            return AutoTokenizer.from_pretrained(str(model_dir), use_fast=False)

def load_model(model_dir_text):
    model_dir = Path(model_dir_text)
    tokenizer = load_tokenizer(model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir), torch_dtype=DTYPE)
    model.to(DEVICE).eval()
    model.config.use_cache = True
    model.config.decoder_start_token_id = 0
    model.config.pad_token_id = 0
    model.config.eos_token_id = 1
    model.generation_config.decoder_start_token_id = 0
    model.generation_config.pad_token_id = 0
    model.generation_config.eos_token_id = 1
    model.generation_config.use_cache = True
    return tokenizer, model

@torch.no_grad()
def generate_one(model_dir, source, beams=4, max_new_tokens=180, length_penalty=1.0):
    tokenizer, model = load_model(str(model_dir))
    batch = tokenizer(
        [source],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)
    generate_kwargs = dict(
        **batch,
        max_new_tokens=int(max_new_tokens),
        num_beams=max(1, int(beams)),
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        length_penalty=float(length_penalty),
        early_stopping=int(beams) > 1,
        use_cache=True,
    )
    ids = model.generate(**generate_kwargs)
    summary = normalize_text(tokenizer.batch_decode(ids, skip_special_tokens=True)[0])
    if not summary:
        retry_ids = model.generate(
            **batch,
            max_new_tokens=int(max_new_tokens),
            num_beams=1,
            do_sample=False,
            use_cache=True,
        )
        summary = normalize_text(tokenizer.batch_decode(retry_ids, skip_special_tokens=True)[0])
        if not summary:
            raw = tokenizer.batch_decode(retry_ids, skip_special_tokens=False)[0]
            summary = f"[EMPTY GENERATION] token_ids={retry_ids[0][:40].tolist()} raw={raw!r}"
        del retry_ids
    del batch, ids, model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    return summary

def compare_models(source, beams, max_new_tokens, length_penalty):
    try:
        source = normalize_text(source)
        if len(source.split()) < 30:
            metrics = [["Nhap bai dai hon", 0, 0, 0, 0, "0/0", 0, 0, 0]]
            return "", "", metrics, []

        start = time.time()
        sft_summary = generate_one(STAGE2_MODEL_DIR, source, beams, max_new_tokens, length_penalty)
        sft_elapsed = time.time() - start

        start = time.time()
        ppo_summary = generate_one(STAGE3_MODEL_DIR, source, beams, max_new_tokens, length_penalty)
        ppo_elapsed = time.time() - start

        rows = [
            metric_row("Stage 2 SFT", source, sft_summary, sft_elapsed),
            metric_row("Stage 3 PPO", source, ppo_summary, ppo_elapsed),
        ]
        metric_cols = ["model", "words", "length_ratio", "number_recall", "number_hallucination", "kept_numbers", "new_numbers", "score", "time_sec"]
        metrics = [[row[col] for col in metric_cols] for row in rows]
        numbers = numbers_comparison(source, sft_summary, ppo_summary)
        return sft_summary, ppo_summary, metrics, numbers
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        message = f"Loi khi chay demo:\n{type(e).__name__}: {e}\n\n{tb[-3500:]}"
        return message, message, [], []

def load_example(choice):
    item = EXAMPLE_MAP.get(choice, {})
    return item.get("source", ""), item.get("reference", "")

def clear_all():
    return "", "", "", "", [], []


## 5. Giao diện Gradio

In [ ]:
CSS = """
.gradio-container { max-width: 1400px !important; }
#model-paths { font-size: 12px; color: #475569; }
textarea { font-size: 14px !important; line-height: 1.55 !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="blue"), css=CSS, title="Compare ViT5 SFT vs PPO") as demo:
    gr.Markdown("# So sánh tóm tắt: Stage 2 SFT vs Stage 3 PPO")
    gr.HTML(f"""
    <div id='model-paths'>
      <b>Stage 2 SFT:</b> {STAGE2_MODEL_DIR}<br>
      <b>Stage 3 PPO:</b> {STAGE3_MODEL_DIR}<br>
      <b>Runtime:</b> {DEVICE}
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            example = gr.Dropdown(
                choices=EXAMPLE_CHOICES,
                value=EXAMPLE_CHOICES[0] if EXAMPLE_CHOICES else None,
                label="Bai mau",
                interactive=True,
            )
            source = gr.Textbox(label="Noi dung bai bao", lines=18, placeholder="Dan noi dung bai bao tieng Viet vao day...")
            reference = gr.Textbox(label="Tom tat tham chieu (neu co trong dataset)", lines=4, interactive=False)
            with gr.Row():
                load_btn = gr.Button("Nap bai mau")
                run_btn = gr.Button("So sanh output", variant="primary")
                clear_btn = gr.Button("Xoa")
            with gr.Accordion("Cai dat generate", open=False):
                beams = gr.Slider(1, 8, value=4, step=1, label="Beam search")
                max_new_tokens = gr.Slider(64, 384, value=180, step=16, label="Max new tokens")
                length_penalty = gr.Slider(0.6, 1.6, value=1.0, step=0.1, label="Length penalty")

        with gr.Column(scale=7):
            with gr.Row():
                sft_output = gr.Textbox(label="Output Stage 2 SFT", lines=10, show_copy_button=True)
                ppo_output = gr.Textbox(label="Output Stage 3 PPO", lines=10, show_copy_button=True)
            metrics = gr.Dataframe(
                headers=["model", "words", "length_ratio", "number_recall", "number_hallucination", "kept_numbers", "new_numbers", "score", "time_sec"],
                datatype=["str", "number", "number", "number", "number", "str", "number", "number", "number"],
                label="Bang so sanh nhanh",
                wrap=True,
            )
            numbers = gr.Dataframe(
                headers=["number", "in_source", "stage2_sft", "stage3_ppo"],
                datatype=["str", "str", "str", "str"],
                label="Doi chieu so lieu trong source va output",
                wrap=True,
            )

    load_btn.click(load_example, inputs=example, outputs=[source, reference])
    example.change(load_example, inputs=example, outputs=[source, reference])
    run_btn.click(compare_models, inputs=[source, beams, max_new_tokens, length_penalty], outputs=[sft_output, ppo_output, metrics, numbers])
    clear_btn.click(clear_all, outputs=[source, reference, sft_output, ppo_output, metrics, numbers])

demo.queue(max_size=4).launch(share=IN_COLAB, show_error=True)
